# Đánh giá baseline marketing — ChatGPT qua **Azure OpenAI** (gpt-4o / gpt-4o-mini / gpt-5.x…)

Notebook **song song** với `marketing_model_eval.ipynb`, **cùng metric** (Faithfulness + Expansion + Marketing Vibe) và **cùng schema CSV** → có thể nối thẳng vào `eval_summary.csv` để so sánh ba nhóm:

1. **Baseline (notebook này)** — ChatGPT qua **Azure OpenAI** (Azure-only, KHÔNG dùng `api.openai.com`).
2. **Open-source local** — Qwen / Gemma… qua LM Studio / Ollama (`marketing_model_eval.ipynb`).
3. **Fine-tuned** — model đã fine-tune (cùng pipeline).

**Dữ liệu:** `dataset/high_quality_mock_cases_100.json`.

**Sinh bài (baseline):** Azure OpenAI — `BASELINE_AZURE_ENDPOINT`, `BASELINE_AZURE_API_KEY`, `BASELINE_MODEL_ID` (deployment name, vd `gpt-4o-mini`).

**Judge:** Azure OpenAI **TÁCH RIÊNG** (dùng `JUDGE_AZURE_*` riêng để tránh self-eval bias; fallback `AZURE_OPENAI_*`).

**Output:** CSV mỗi run tại `results/runs/…`, tổng hợp append vào `results/high_quality_mock_cases_100.eval_summary.csv` — **cùng file** với notebook open-source để dễ so sánh.

In [ ]:
# %pip install -q deepeval openai pandas python-dotenv

In [1]:
import os
from pathlib import Path
from typing import Optional, List, Dict
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
load_dotenv(_dotenv_path, override=True, encoding="utf-8")
ROOT = Path(_dotenv_path).resolve().parent if _dotenv_path else Path.cwd().resolve()

# ===== Hyperparameters =====
DATASET_REL = "dataset/fnb_dataset_test.json"
SUMMARY_REL = "results/fnb_dataset_test.eval_summary.csv"
RUNS_REL = "results/runs"

OUTPUT_FROM_CSV: Optional[Path] = None

def _env(k: str) -> str:
    return (os.getenv(k) or "").strip()

# ---- Baseline (sinh bài) — Azure-only ----
# Deployment name trên Azure (KHÔNG phải model name của OpenAI public).
BASELINE_MODEL = (os.getenv("BASELINE_MODEL") or "gpt-4o-mini").strip()
GENERATION_TEMPERATURE = 0.4

BASELINE_AZURE_ENDPOINT = (_env("BASELINE_AZURE_ENDPOINT") or _env("AZURE_OPENAI_ENDPOINT")).rstrip("/")
BASELINE_AZURE_API_KEY = _env("BASELINE_AZURE_API_KEY") or _env("AZURE_OPENAI_API_KEY")
BASELINE_AZURE_API_VERSION = _env("BASELINE_AZURE_API_VERSION") or _env("OPENAI_API_VERSION") or "2024-08-01-preview"
if not (BASELINE_AZURE_ENDPOINT and BASELINE_AZURE_API_KEY):
    raise RuntimeError("Baseline cần BASELINE_AZURE_ENDPOINT + BASELINE_AZURE_API_KEY (hoặc AZURE_OPENAI_*).")
BASELINE_BACKEND = "azure"

# ---- Judge — Azure-only, tách biệt khỏi baseline ----
JUDGE_MODEL = (os.getenv("JUDGE_MODEL") or os.getenv("OPENAI_MODEL") or "gpt-4o-mini").strip()
JUDGE_AZURE_ENDPOINT = (_env("JUDGE_AZURE_ENDPOINT") or _env("AZURE_OPENAI_ENDPOINT")).rstrip("/")
JUDGE_AZURE_API_KEY = _env("JUDGE_AZURE_API_KEY") or _env("AZURE_OPENAI_API_KEY")
JUDGE_AZURE_API_VERSION = _env("JUDGE_AZURE_API_VERSION") or _env("OPENAI_API_VERSION") or "2024-08-01-preview"
if not (JUDGE_AZURE_ENDPOINT and JUDGE_AZURE_API_KEY):
    raise RuntimeError("Judge cần JUDGE_AZURE_ENDPOINT + JUDGE_AZURE_API_KEY (hoặc AZURE_OPENAI_*).")
JUDGE_BACKEND = "azure"

# Tương thích DeepEval flow: bắt buộc Azure mode.
os.environ["USE_AZURE_OPENAI"] = "true"
os.environ.setdefault("OPENAI_API_VERSION", JUDGE_AZURE_API_VERSION)
os.environ.pop("USE_OPENAI_MODEL", None)
os.environ["OPENAI_MODEL"] = JUDGE_BACKEND

# ⚠️ Cảnh báo self-eval: baseline và judge nên dùng deployment KHÁC nhau.
if BASELINE_MODEL.lower() == JUDGE_MODEL.lower():
    print(f"⚠️  CẢNH BÁO: baseline ({BASELINE_MODEL}) = judge ({JUDGE_MODEL}). Có self-eval bias.")

EVAL_PROGRESS_CHUNK = 10

DATASET_PATH = ROOT / DATASET_REL
SUMMARY_PATH = ROOT / SUMMARY_REL
RUNS_DIR = ROOT / RUNS_REL

print("Baseline (Azure):", BASELINE_MODEL, "@", BASELINE_AZURE_ENDPOINT)
print("Judge    (Azure):", JUDGE_MODEL, "@", JUDGE_AZURE_ENDPOINT)

Baseline (Azure): md-gpt-5.4-mini @ https://vqnhan-poc.openai.azure.com
Judge    (Azure): gpt-5.4 @ https://vqnhan-poc.openai.azure.com


In [2]:
# Verify nhanh: Judge + Baseline — Azure-only
from openai import AzureOpenAI

_hello_prompt = "Xin chào, bạn là ai"

_judge = AzureOpenAI(
    azure_endpoint=JUDGE_AZURE_ENDPOINT,
    api_key=JUDGE_AZURE_API_KEY,
    api_version=JUDGE_AZURE_API_VERSION,
)
_judge_model_id = JUDGE_MODEL

_r = _judge.chat.completions.create(
    model=_judge_model_id,
    messages=[{"role": "user", "content": _hello_prompt}],
)
print("Judge:", (_r.choices[0].message.content or "").strip())

Judge: Xin chào! Tôi là trợ lý AI của bạn. Tôi có thể giúp trả lời câu hỏi, giải thích kiến thức, viết nội dung, dịch thuật, hỗ trợ lập trình và nhiều việc khác.

Bạn muốn tôi giúp gì hôm nay?


In [3]:
_baseline = AzureOpenAI(
    azure_endpoint=BASELINE_AZURE_ENDPOINT,
    api_key=BASELINE_AZURE_API_KEY,
    api_version=BASELINE_AZURE_API_VERSION,
)

_r2 = _baseline.chat.completions.create(
    model=BASELINE_MODEL,
    messages=[{"role": "user", "content": _hello_prompt}],
)
print("Baseline:", (_r2.choices[0].message.content or "").strip())

Baseline: Xin chào! Mình là ChatGPT, một trợ lý AI có thể giúp bạn trả lời câu hỏi, viết nội dung, dịch thuật, giải thích kiến thức, và hỗ trợ nhiều việc khác.

Bạn muốn mình giúp gì hôm nay?


In [4]:
import json
import re
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from dotenv import find_dotenv, load_dotenv
from metrics import (
    JudgeLLM,
    FaithfulnessEvaluator,
    ExpansionQualityEvaluator,
    MarketingVibeEvaluator,
    run_batch,
)

load_dotenv(find_dotenv(usecwd=True), override=True, encoding="utf-8")

judge_llm = JudgeLLM(_judge, _judge_model_id)
print(f"JudgeLLM: {judge_llm.get_model_name()} (judge_backend={JUDGE_BACKEND})")

JudgeLLM: gpt-5.4 (judge_backend=azure)


## Load dataset

In [5]:
def load_cases(json_path: Path, override_csv: Optional[Path]) -> List[Dict[str, str]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Dataset phải là mảng JSON.")
    cases = []
    for idx, row in enumerate(data):
        cases.append(
            {
                "case_id": str(idx + 1),
                "instruction": str(row["instruction"]),
                "input_title": str(row["title"]),
                "seed_content": str(row["seed"]),
                "actual_output": "",  # baseline notebook: luôn sinh mới qua ChatGPT
            }
        )
    if override_csv and override_csv.is_file():
        df_o = pd.read_csv(override_csv)
        if "case_id" not in df_o.columns or "actual_output" not in df_o.columns:
            raise ValueError("CSV override cần cột: case_id, actual_output")
        m = dict(zip(df_o["case_id"].astype(str), df_o["actual_output"].astype(str)))
        for c in cases:
            if c["case_id"] in m:
                c["actual_output"] = m[c["case_id"]]
    return cases


cases = load_cases(DATASET_PATH, OUTPUT_FROM_CSV)
print(f"Cases: {len(cases)} | dataset: {DATASET_PATH.name} | BASELINE_MODEL_ID: {BASELINE_MODEL}")

Cases: 100 | dataset: fnb_dataset_test.json | BASELINE_MODEL_ID: md-gpt-5.4-mini


### Sinh `actual_output` qua ChatGPT (baseline)

Chạy **sau** ô load `cases`, **trước** ô `run_batch`. Dùng client baseline (`_baseline`) đã khởi tạo ở ô verify.

In [6]:
def build_user_prompt(title: str, seed: str) -> str:
    return (
        f"Viết bài marketing Markdown từ tiêu đề và mồi:\n\n"
        f"Tiêu đề: {title}\n\n"
        f"Mồi:\n{seed}"
    )

_gen_t0 = time.perf_counter()
for idx, c in enumerate(cases, start=1):
    if c.get("actual_output"):
        continue  # đã có từ override CSV
    r = _baseline.chat.completions.create(
        model=BASELINE_MODEL,
        messages=[
            {"role": "system", "content": c["instruction"]},
            {"role": "user", "content": build_user_prompt(c["input_title"], c["seed_content"])},
        ],
        temperature=GENERATION_TEMPERATURE,
    )
    c["actual_output"] = (r.choices[0].message.content or "").strip()
    if idx % EVAL_PROGRESS_CHUNK == 0 or idx == len(cases):
        print(f"Sinh bài {idx}/{len(cases)} — elapsed {time.perf_counter() - _gen_t0:.1f}s", flush=True)
print("Hoàn tất sinh bài.")

Sinh bài 10/100 — elapsed 31.4s
Sinh bài 20/100 — elapsed 56.0s
Sinh bài 30/100 — elapsed 86.4s
Sinh bài 40/100 — elapsed 118.4s
Sinh bài 50/100 — elapsed 144.0s
Sinh bài 60/100 — elapsed 170.9s
Sinh bài 70/100 — elapsed 202.7s
Sinh bài 80/100 — elapsed 232.8s
Sinh bài 90/100 — elapsed 264.3s
Sinh bài 100/100 — elapsed 295.0s
Hoàn tất sinh bài.


In [7]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", BASELINE_MODEL).strip("_") or "baseline"
csv_path = RUNS_DIR / f"baseline_{safe_model}_{RUN_ID}.csv"

df_scores = run_batch(cases, judge_llm, progress_chunk=EVAL_PROGRESS_CHUNK)

base = pd.DataFrame([{**c} for c in cases])
df_out = pd.concat(
    [base.reset_index(drop=True), df_scores.drop(columns=["case_id"]).reset_index(drop=True)],
    axis=1,
)
df_out.insert(0, "run_id", RUN_ID)
df_out.insert(1, "judge_backend", JUDGE_BACKEND)
df_out.insert(2, "judge_model", JUDGE_MODEL)
# Cùng schema cột với notebook open-source để tổng hợp chung.
df_out.insert(3, "local_llm_backend", BASELINE_BACKEND)
df_out.insert(4, "local_model_id", BASELINE_MODEL)
df_out.insert(5, "local_openai_base", BASELINE_AZURE_ENDPOINT)
df_out.insert(6, "model_role", "baseline")

df_out.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {csv_path}")

c:\Users\vqnhan\AppData\Local\Programs\Python\Python314\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Chunk 1–10/100 hoàn thành trong 131.6s


Chunk 11–20/100 hoàn thành trong 129.5s


Chunk 21–30/100 hoàn thành trong 123.9s


Chunk 31–40/100 hoàn thành trong 125.3s


...

Chunk 51–60/100 hoàn thành trong 167.1s


Chunk 61–70/100 hoàn thành trong 149.4s


Chunk 71–80/100 hoàn thành trong 131.3s


Chunk 81–90/100 hoàn thành trong 123.4s


Chunk 91–100/100 hoàn thành trong 125.1s
Đã lưu: D:\Github\mcs-train-content-model\results\runs\baseline_md-gpt-5.4-mini_20260519T155957Z.csv


In [8]:
summary_cols = ["faithfulness_combined", "expansion_combined", "vibe_combined"]
means = df_scores[summary_cols].mean().to_dict()
overall = float(np.mean([means[c] for c in summary_cols]))

now = datetime.now(timezone.utc).isoformat()
summary_row = {
    "run_id": RUN_ID,
    "judge_backend": JUDGE_BACKEND,
    "judge_model": JUDGE_MODEL,
    "local_llm_backend": BASELINE_BACKEND,
    "local_model_id": BASELINE_MODEL,
    "local_openai_base": BASELINE_AZURE_ENDPOINT,
    "model_role": "baseline",
    "dataset_file": str(DATASET_PATH.relative_to(ROOT)),
    "n_cases": len(cases),
    "csv_file": str(csv_path.relative_to(ROOT)),
    "faithfulness_combined": means["faithfulness_combined"],
    "expansion_combined": means["expansion_combined"],
    "vibe_combined": means["vibe_combined"],
    "overall_mean": overall,
    "last_updated_utc": now,
}

new_df = pd.DataFrame([summary_row])
if SUMMARY_PATH.is_file():
    prev_df = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
    summary_out = pd.concat([prev_df, new_df], ignore_index=True)
else:
    summary_out = new_df
summary_out.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"Đã cập nhật tổng hợp: {SUMMARY_PATH}")
pd.DataFrame([summary_row]).T

Đã cập nhật tổng hợp: D:\Github\mcs-train-content-model\results\fnb_dataset_test.eval_summary.csv


,0
run_id,20260519T155957Z
judge_backend,azure
judge_model,gpt-5.4
local_llm_backend,azure
local_model_id,md-gpt-5.4-mini
local_openai_base,https://vqnhan-poc.openai.azure.com
model_role,baseline
dataset_file,dataset\fnb_dataset_test.json
n_cases,100
csv_file,results\runs\baseline_md-gpt-5.4-mini_20260519...


In [9]:
# So sánh nhanh 3 nhóm: baseline (ChatGPT) vs open-source (Qwen/Gemma) vs fine-tuned.
# Nếu cột `model_role` chưa có với các run cũ, NaN sẽ được điền 'unknown'.
df_all = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
if "model_role" not in df_all.columns:
    df_all["model_role"] = "unknown"
df_all["model_role"] = df_all["model_role"].fillna("unknown")
cols = ["model_role", "local_model_id", "run_id",
        "faithfulness_combined", "expansion_combined", "vibe_combined", "overall_mean"]
df_all[cols].sort_values(["model_role", "local_model_id", "run_id"])

,model_role,local_model_id,run_id,faithfulness_combined,expansion_combined,vibe_combined,overall_mean
0,baseline,md-gpt-5.4-mini,20260519T155957Z,0.682556,0.7491,0.678622,0.703426
